In [39]:
import numpy as np
import pandas as pd
import cftime
import re
import time
import gc
import os
import math
import glob
import datetime
import sys
from scipy.ndimage import label, generate_binary_structure

sys.path.insert(1, '../')
from functions import *
from functions_stage2 import (get_rainfall_cube_subsection, maybe_diagnose, analyse_peak_event)
verbose=True
from processing_functions import find_max_precip_location_new

# ── Config ────────────────────────────────────────────────────────────────────
from config import RAINFALL_CSV_DIR, CATCHMENT_LOOKUP_DICT, OUT_DIR, CATCHMENTS, ENSEMBLE_MEMBERS 

FLOOD_ROOT = "/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/"
RAIN_ROOT = "/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/"
RAIN_CUBE_ROOT = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/"

SM_ROOT =   '/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/'

WET_THRESH   = 0.1          # mm/hr; an hour counts as wet above this
FLOOD_DEPTHS = [10, 30]     # cm

In [47]:
def load_flood_arrays(catchment_num, ens_num, mask, catchment_poly,
                      depths=FLOOD_DEPTHS, n_events=None):
    """
    Load flooded area and volume for one catchment and ensemble member.

    Returns
    -------
    flood_cubes : {depth: {'area': cube, 'vol': cube}}
    flood_numpy : {depth: {'area': ndarray, 'vol': ndarray}}
        Arrays are (event, y, x). Axis 0 is event_num - 1.
    """
    ens_dir = f"{FLOOD_ROOT}Catchment_{catchment_num}/Ens{ens_num}_{catchment_num}/"

    flood_cubes, flood_numpy = {}, {}
    for depth in depths:
        flood_cubes[depth] = {}
        flood_numpy[depth] = {}
        # dictionary key -> variable name used in the filename
        for key, name in [('area', 'area'), ('vol', 'volume')]:
            fp = (f"{ens_dir}/{depth}cm/flooded_{name}_5km_total_"
                  f"Ens{ens_num}_{catchment_num}_{depth}cm.nc")
            cube = prepare_flood_cube(load_3d_cube(fp))
            cube.data = np.where(mask, cube.data, np.nan)
            cube = filter_closer_to_catchment(cube, catchment_poly, plot=False)

            flood_cubes[depth][key] = cube
            flood_numpy[depth][key] = cube.data

    # sanity checks
    shapes = {arr.shape for d in flood_numpy.values() for arr in d.values()}
    assert len(shapes) == 1, f"Flood arrays have inconsistent shapes: {shapes}"
    if n_events is not None:
        n_flood = next(iter(shapes))[0]
        assert n_flood == n_events, \
            f"Flood file has {n_flood} events but the events CSV has {n_events}"

    return flood_cubes, flood_numpy


def load_rainfall_events(catchment_name, ens_num):
    """"""
    rainfall_events  = pd.read_csv(RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")
    return rainfall_events  

def add_hydro_year(events):
    """Number events 1..N and assign hydro year (Dec events belong to the next year)."""
    events['event_num']  = range(1, len(events) + 1)
    events['hydro_year'] = events['start_year'] + (events['start_month'] == 12).astype(int)
    return events

def find_wet_onset(values, time_coord, wet_thresh=WET_THRESH):
    """First/last wet hour (index into the cube) and onset date, or None if never wet."""
    wet = np.flatnonzero(values > wet_thresh)
    if wet.size == 0:
        return None
    first, last = int(wet[0]), int(wet[-1])
    t = time_coord.units.num2date(time_coord.points[first])
    return first, last, (t.year, t.month, t.day)

def sm_day_before(sm_data, day_index, onset_day):
    """(y, x) SM field for the day before onset_day, or NaNs if unavailable."""
    idx = day_index.get(onset_day)
    if idx is None or idx == 0:
        return np.full(sm_data.shape[1:], np.nan)
    return sm_data[idx - 1]

def build_catchment_mask(full_rain_cube, CATCHMENT_POLY):
    """"""
    
    FULL_MASK_2D   = mask_cube_with_catchment_full_grid(full_rain_cube[0], CATCHMENT_POLY, method='center_point')
    year_cube, mask_x_offset, mask_y_offset = subset_cube_to_bbox(full_rain_cube, CATCHMENT_POLY, buffer=0)
    ny_sub = year_cube.shape[1]
    nx_sub = year_cube.shape[2]
    mask_2d_sub = FULL_MASK_2D[mask_y_offset:mask_y_offset + ny_sub, 
                                mask_x_offset:mask_x_offset + nx_sub]
    del full_rain_cube, year_cube
    return FULL_MASK_2D, mask_2d_sub


def load_soil_moisture(events_in_year, ens_num, hydro_year, catchment_poly, mask):
    """
    Load daily soil moisture for one hydro year (1 Dec to 30 Nov), masked and cropped
    to the catchment. If any event starts on 1 December, the previous year's file is
    prepended so the "day before" lookup works.

    Returns a SoilMoisture with: cube, data (t, y, x), times, day_index {(y, m, d): i}.
    """
    sm_dir  = SM_ROOT + f'Ens_{ens_num}/'
    sm_file = glob.glob(f"{sm_dir}/r001i1p*****_{hydro_year-1}1201-{hydro_year}1130_mrso.nc")[0]
    sm      = iris.load(sm_file)[0]

    needs_extra_sm = ((events_in_year['start_month'] == 12) & (events_in_year['start_day'] == 1)).any()
    if needs_extra_sm:
        prev_files = glob.glob(f"{sm_dir}/r001i1p*****_{hydro_year-2}1201-{hydro_year-1}1130_mrso.nc")
        if prev_files:
            prev_sm = iris.load(prev_files[0])[0]
            prev_sm.attributes = {}
            sm.attributes = {}
            sm = iris.cube.CubeList([prev_sm, sm]).concatenate_cube()
        else:
            print(f"  WARNING: prior SM file for {hydro_year-2}-{hydro_year-1} not found")

    sm.data = np.where(mask, sm.data, np.nan)
    sm      = filter_closer_to_catchment(sm, catchment_poly, plot=False)

    times     = np.array([cell.point for cell in sm.coord('time').cells()])
    day_index = {(c.year, c.month, c.day): i for i, c in enumerate(times)}

    return SoilMoisture(cube=sm, data=sm.data, times=times, day_index=day_index)

def plot_sm_check(sm_data, sm_time_arr, sm_day_idx, anchor_key, yi, xi,
                  values, time_coord, n=10):
    idx = sm_day_idx[anchor_key]
    lo, hi = max(idx - n, 0), min(idx + n + 1, len(sm_time_arr))
    offsets = np.arange(lo, hi) - idx
    sm_series = sm_data[lo:hi, yi, xi]

    # daily rainfall at the peak cell (only covers the event window)
    rain_by_day = {}
    for p, v in zip(time_coord.points, values):
        d = time_coord.units.num2date(p)
        k = (d.year, d.month, d.day)
        rain_by_day[k] = rain_by_day.get(k, 0) + (0 if np.isnan(v) else v)
    rain = [rain_by_day.get((c.year, c.month, c.day), 0) for c in sm_time_arr[lo:hi]]

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(offsets, sm_series, 'o-', label='SM at cell')
    ax.axvline(0,  color='k', ls='--', label='first wet day')
    ax.axvline(-1, color='r', ls='--', label='day used')
    ax.set_xlabel('days from first wet day'); ax.set_ylabel('SM')
    ax2 = ax.twinx(); ax2.bar(offsets, rain, alpha=0.3, color='tab:blue')
    ax2.set_ylabel('rain at cell (event window)')
    ax.legend(); plt.show()
    
def extract_at_cell(yi, xi, event_cube, flood_numpy, event_idx, sm, prefix):
    """
    Collect rainfall, antecedent soil moisture and flood values at one grid cell.

    The same function is used for each "anchor" cell (e.g. the rainfall-peak cell and
    the flood-peak cell). Every output key starts with `prefix` so the two sets of
    results can sit side by side in one row.

    Parameters
    ----------
    yi, xi      : cell indices on the cropped catchment grid (shared by the rainfall,
                  SM and flood arrays)
    event_cube  : hourly rainfall cube for this event, (time, y, x). It starts one
                  hour before the CSV event start, so index 0 is that lead-in hour.
    flood_numpy : {depth: {'area': array, 'vol': array}}, arrays are (event, y, x)
    event_idx   : position of this event along the flood arrays' first axis
                  (event_num - 1)
    sm          : SoilMoisture bundle for this hydro year
    prefix      : e.g. 'rainpeak' or 'floodpeak'
    """
    out = {f'{prefix}_y_idx': yi, f'{prefix}_x_idx': xi}

    # ---- Rainfall at this cell ---------------------------------------------------
    # Hourly series over the whole event window (including the lead-in hour and any
    # dry hours at the start/end). Masked values become NaN.
    series = np.ma.filled(event_cube.data[:, yi, xi], np.nan)
    out[f'{prefix}_rain_series'] = series.astype(np.float32)
    out[f'{prefix}_max_precip']   = round(float(np.nanmax(series)), 3)
    out[f'{prefix}_total_precip'] = round(float(np.nansum(series)), 3)

    # ---- Event onset and trimmed rainfall ------------------------------------------
    # find_wet_onset returns the first and last hours above the wet threshold, plus
    # the calendar date of the first wet hour. That date anchors the SM lookup.
    # first/last are positions in the event cube (0 = the lead-in hour).
    onset = find_wet_onset(series, event_cube.coord('time'))

    if onset is None:
        # Never above the wet threshold at this cell: keep the same columns,
        # but fill them with empty/NaN values so the dataframe stays rectangular.
        out[f'{prefix}_no_wet_hours'] = True
        out[f'{prefix}_first_wet_idx'] = np.nan
        out[f'{prefix}_trimmed_duration_hrs'] = 0
        out[f'{prefix}_rain_series_trimmed'] = np.array([], dtype=np.float32)
        out[f'{prefix}_sm_day_before_onset'] = np.nan
    else:
        first, last, anchor = onset
        out[f'{prefix}_no_wet_hours'] = False
        out[f'{prefix}_first_wet_idx'] = first
        out[f'{prefix}_trimmed_duration_hrs'] = last - first + 1

        # Rainfall from the first to the last wet hour, inclusive, so the series
        # starts and ends on values above the wet threshold (dry hours inside it stay).
        out[f'{prefix}_rain_series_trimmed'] = series[first:last + 1].astype(np.float32)

        # ---- Soil moisture the day before the first wet day ------------------------
        # Returns a (y, x) field, so the value is taken at this cell.
        sm_field = sm_day_before(sm.data, sm.day_index, anchor)
        out[f'{prefix}_sm_day_before_onset'] = round(float(sm_field[yi, xi]), 2)

    # ---- Flooding at this cell for this event -------------------------------------
    # One value per flood depth and metric. Axis 0 of the flood arrays is the event.
    for depth, d in flood_numpy.items():
        out[f'{prefix}_{depth}cm_area']   = round(float(d['area'][event_idx, yi, xi]), 2)
        out[f'{prefix}_{depth}cm_volume'] = round(float(d['vol'][event_idx, yi, xi]), 1)

    return out


def get_rainfall_data_from_climate_data(hydro_year, ens_num, row, mask, poly):

    rainfall_cube_dir = RAIN_CUBE_ROOT + f"{ens_num}/"

    # -------------------------
    # Get rainfall data for the hydro-year this event happened in
    # -------------------------
    year_cube = get_rainfall_cube_subsection(hydro_year, ens_num, rainfall_cube_dir, 
                                         row.start_indices-1,row.stop_indices)
    year_cube.data = np.where(mask, year_cube.data, np.nan)

    # -------------------------
    # Get times
    # -------------------------       
    time_coord  = year_cube.coord('time')
    t0 = time_coord.units.num2date(np.floor(time_coord.points[0]))
    t_end = time_coord.units.num2date(time_coord.points[-1])
    event_start = cftime.Datetime360Day(row.start_year, row.start_month, row.start_day, row.start_hour)

    # Check timings make sense
    event_start_within_cube = t0 <= event_start <= t_end

    if not event_start_within_cube:
        time_mismatch = True
        print("TIME MISMACH")
    else:
        time_mismatch = False     

    return year_cube, time_coord, time_mismatch

def get_rainfall_and_floding_at_peak_cells(pk, prefix, year_cube, flood_numpy, sm,  x_offset, y_offset, event_idx, result = None):
    if result == None:
        result = {}

    result[f'{prefix}_found'] = pk is not None
    #if pk is None:
    #    continue                                  # keep the event, with NaNs for this anchor
    result[f'{prefix}_peak_value'] = round(pk['peak_value'],1)
    result[f'{prefix}_x_global']   = pk['x_idx'] + x_offset
    result[f'{prefix}_y_global']   = pk['y_idx'] + y_offset
    result[f'{prefix}_x_coord'] = float(year_cube.coord('projection_x_coordinate').points[pk['x_idx']])
    result[f'{prefix}_y_coord'] = float(year_cube.coord('projection_y_coordinate').points[pk['y_idx']])
    
    if prefix =='rainpeak':
        result[f'{prefix}_t_local']     = pk['t_local']
    
    result.update(extract_at_cell(pk['y_idx'], pk['x_idx'], year_cube, flood_numpy, event_idx, sm, prefix))

    return result                


def find_peak_cell(field, min_value=0.0):
    """Peak of a (y, x) or (t, y, x) array. Returns None if there is no peak above min_value."""
    arr = np.array(np.ma.filled(field, np.nan), dtype=float)   # copy, so the cube is untouched
    arr[arr >= 1e19] = np.nan                                  # fill values

    if np.all(np.isnan(arr)) or np.nanmax(arr) <= min_value:
        return None

    idx = np.unravel_index(np.nanargmax(arr), arr.shape)
    out = {'y_idx': int(idx[-2]), 'x_idx': int(idx[-1]), 'peak_value': float(arr[idx])}
    if arr.ndim == 3:
        out['t_local'] = int(idx[0])
    return out

def process_event(row, ens_num, hydro_year):
    result = {
        'ens': ens_num, 'hydro_year': hydro_year, 'event_num': int(row.event_num),
        'start_idx': int(row.start_indices), 'stop_idx': int(row.stop_indices),
        'max_precip_from_csv': float(row.peaks),
        'start_year': int(row.start_year), 'start_month': int(row.start_month),
        'start_day': int(row.start_day),   'start_hour': int(row.start_hour),
        'event_durations': int(row.event_durations)}

    # Get event number
    event_num = int(row.event_num)
    event_idx = event_num - 1

    # Get rainfall data for the hydro-year this event happened in
    # Check whether there is a time mismatch (i.e.)
    year_cube, time_coord, time_mismatch = get_rainfall_data_from_climate_data(hydro_year, ens_num, row, mask, CATCHMENT_POLY)
    result['time_mismatch'] = time_mismatch

    # Subset the rainfall data to the catchment
    # Get a reference for the spatial offset of the X and Y coordinates (i.e. to find equivalent location in non-subsetted-cube)
    year_cube, x_offset, y_offset = subset_cube_to_bbox(year_cube, CATCHMENT_POLY, buffer=0)

    # Check flood, rainfall and soil moisture data are all the same size
    # assert flood_numpy[10]['vol'].shape[1:] == year_cube.shape[1:]  == sm.data.shape[1:]    
    
    # Find the location of the cells where the peak rainfall/flood occurs
    rain_pk  = find_peak_cell(year_cube.data)                       # (t, y, x)
    flood_pk = find_peak_cell(flood_numpy[10]['area'][event_idx])     # (y, x)

    # Get the rainfall data at the cell with the peak rainfall and flooding 
    result = get_rainfall_and_floding_at_peak_cells (rain_pk, "rainpeak", year_cube, flood_numpy, sm,  x_offset, y_offset, 
                                         event_idx, result)
    result = get_rainfall_and_floding_at_peak_cells (flood_pk, "floodpeak", year_cube, flood_numpy, sm,  x_offset, y_offset,
                                         event_idx, result)
    
    ## Add variables describing whether the peak in the rainfall data and the flood data occur in the same cell
    if rain_pk and flood_pk:
        result['peaks_same_cell'] = (rain_pk['y_idx'], rain_pk['x_idx']) == (flood_pk['y_idx'], flood_pk['x_idx'])
        # Find the distance (using the hypotenuse) between the two peak cells
        result['peaks_dist_cells'] = float(np.hypot(rain_pk['y_idx'] - flood_pk['y_idx'],
                                                    rain_pk['x_idx'] - flood_pk['x_idx']))    
        
   # Get rainfall peak start times
    t = time_coord.units.num2date(time_coord.points[result['rainpeak_t_local']])
    result['rainfall_peak_day'] = cftime.Datetime360Day(t.year, t.month, t.day)
    result['rainfall_peak_day_hydro_day_360'] = ((result['rainfall_peak_day'].month % 12) * 30 + result['rainfall_peak_day'].day)
    result['rainfall_peak_day_daY_360'] = ((result['rainfall_peak_day'].month - 1) * 30 + result['rainfall_peak_day'].day)
    result['t_global'] =  int(result['rainpeak_t_local'] + row.start_indices)
     
    # Add variables describing spatial situation around the peak
    result.update(analyse_peak_event(
        result, year_cube, flood_numpy, flood_cubes, CATCHMENT_POLY, boundary_gdf,
        neighbourhood_size=1, threshold_levels=[0.5, 0.6, 0.8], plot=False))
        
    return result

def analyse_peak_event(
    this_event_results,
    year_cube,
    flood_numpy,
    flood_cubes,          # only needed for plotting (iris cubes retain coords for maps)
    CATCHMENT_POLY,       # only needed for plotting
    boundary_gdf,         # only needed for plotting
    neighbourhood_size,
    threshold_levels,
    plot=False
):
    """
    Characterise a single rainfall event's spatial footprint at three
    nested spatial scales, and summarise flood outcomes and antecedent
    wetness conditions (rainfall and soil moisture) over each of those
    same footprints.

    The three spatial scales, from smallest/fixed to largest/event-defined:

    1. Peak cell (yi, xi)
       — the single grid cell where the event's maximum precipitation
         occurred. Used as the indexing anchor for the other two scales,
         and directly for the '..._point' antecedent variables.

    2. Neighbourhood
       — a fixed-size square window of (2*neighbourhood_size + 1) cells
         centred on the peak cell, regardless of how the storm actually
         looked. A crude, storm-agnostic "local area" scale — useful as
         a baseline that doesn't depend on the threshold_levels choice.

    3. Threshold mask / cluster
       — cells where rainfall exceeded `threshold_level * peak_value`.
         `threshold_mask` includes ALL such cells in the catchment (which
         may include unconnected patches elsewhere in the domain that
         happened to also be intense). `cluster_mask` is the subset of
         `threshold_mask` that is spatially CONNECTED to the peak cell
         (via scipy.ndimage.label) — i.e. the actual contiguous storm
         footprint containing the peak, not just "intense cells anywhere".
         This is repeated at each level in `threshold_levels` (e.g. 50%,
         60%, 80% of peak intensity) to see how footprint size and
         antecedent conditions change as you tighten the definition of
         "part of the storm".

    For each of these three scales, the function reports:
      - the rainfall itself (total/n_cells for footprint size and intensity)
      - flood outcomes (area, volume, at 10cm and 30cm depth thresholds)
      - antecedent conditions (from `antecedent_arrays`), summarised as
        the MEAN over the footprint — not the sum, since footprint size
        varies event-to-event and across threshold levels, and we want
        "how wet was this area beforehand" (a rate/state), not "how much
        antecedent rain fell across an area of this size" (which would
        just track footprint size and confound with storm extent).

    Parameters
    ----------
    this_event_results : dict
        Per-event metadata already computed upstream: must contain
        'max_precip', 't_local' (time index of peak within year_cube),
        'y_idx'/'x_idx' (grid location of peak, within year_cube's
        coordinate system), and 'event_num' (1-indexed, used to index
        into the flood arrays' leading (event) dimension).
    year_cube : iris.cube.Cube
        The event-window rainfall cube (time, y, x), already subset to
        the catchment bbox. Only the peak-time slice is used here.
    flood_numpy : dict
        {depth: {'area': ndarray, 'vol': ndarray}} — flood outcome
        arrays, shape (event, y, x), same y/x grid as year_cube.
    neighbourhood_size : int
        Half-width of the fixed neighbourhood window, in cells.
    threshold_levels : list of float
        e.g. [0.5, 0.6, 0.8] — fractions of peak intensity defining the
        storm footprint at each scale.
    plot : bool
        If True, generates a diagnostic figure at the 50% threshold level
        only (for spot-checking specific events, not for bulk runs).

    Returns
    -------
    dict
        Flat dict of scalar results, suffixed by scale
        (neighbourhood_/threshold_..._t50/cluster_..._t50 etc.),
        ready to be merged into `this_event_results`.
    """

    # ── Pull out the essentials for this event ──────────────────────────────
    peak_value = this_event_results['rainpeak_peak_value']
    t_local    = int(this_event_results['rainpeak_t_local'])
    yi         = int(this_event_results['rainpeak_y_idx'])   # peak cell row
    xi         = int(this_event_results['rainpeak_x_idx'])   # peak cell col

    # ── Extract the rainfall field AT THE MOMENT OF PEAK INTENSITY ──────────
    # (This is a single time-slice, not a total over the event — it's what
    # defines the storm's spatial footprint/shape at its most intense moment.)
    peak_slice = year_cube[t_local, :, :]
    rain = peak_slice.data.astype(float)

    # Clean sentinel/fill values so they don't pollute nansum/nanmean below
    rain = np.where((rain == -99999) | (rain > 1e19), np.nan, rain)

    # ── Pre-slice flood arrays down to this event's 2D (y, x) layer ─────────
    # (flood_numpy is 3D: event x y x x — index into the event dimension once,
    # up front, so the threshold loop below doesn't repeat this lookup.)
    flood_arrays = {
        (depth, var): flood_numpy[depth][var][this_event_results['event_num'] - 1, :, :]
        for depth in [10, 30]
        for var in ['area', 'vol']
    }

    # ══════════════════════════════════════════════════════════════════════
    # SCALE 1: Neighbourhood — fixed-size window around the peak cell
    # ══════════════════════════════════════════════════════════════════════
    n  = neighbourhood_size
    y0 = max(0, yi - n);  y1 = min(rain.shape[0], yi + n + 1)
    x0 = max(0, xi - n);  x1 = min(rain.shape[1], xi + n + 1)
    neighbourhood = rain[y0:y1, x0:x1]

    results = {
        # Total rainfall summed across the neighbourhood window
        'neighbourhood_rain_total': float(np.nansum(neighbourhood)),
        # Same, but excluding the peak cell's own value (isolates the
        # contribution of the surrounding cells only)
        'neighbourhood_rain_sum_excl_peak': float(np.nansum(neighbourhood) - peak_value),
        # How many valid (non-NaN, i.e. in-catchment) cells the window
        # actually covered — window can be clipped near catchment edges
        'neighbourhood_n_cells': int(np.sum(~np.isnan(neighbourhood))),
    }

    # Flood outcomes summed over the same neighbourhood window
    for depth in [10, 30]:
        for var in ['area', 'vol']:
            results[f'neighbourhood_flood_{depth}_{var}'] = float(
                np.nansum(flood_arrays[(depth, var)][y0:y1, x0:x1]))


    # ══════════════════════════════════════════════════════════════════════
    # SCALE 2 & 3: Threshold mask and connected cluster, per threshold level
    # ══════════════════════════════════════════════════════════════════════
    for threshold_level in threshold_levels:
        t_suffix = f"_t{int(threshold_level * 100)}"   # e.g. '_t50', '_t60', '_t80'
        threshold_value = peak_value * threshold_level

        # All cells (anywhere in the catchment) whose rainfall meets the
        # threshold — may include disconnected patches unrelated to this storm
        threshold_mask = (rain >= threshold_value) & (~np.isnan(rain))

        # Connected-component labelling: isolates just the patch of
        # threshold-exceeding cells that is spatially contiguous with the
        # peak cell — i.e. the actual storm footprint, not incidental
        # intense cells elsewhere in the catchment
        labeled, _ = label(threshold_mask)
        peak_label = labeled[yi, xi]
        cluster_mask = (labeled == peak_label)

        # Rainfall stats: footprint size (n_cells) and total intensity,
        # for both the loose threshold set and the tight connected cluster
        results[f'threshold_rain_total{t_suffix}'] = float(np.nansum(rain[threshold_mask]))
        results[f'threshold_n_cells{t_suffix}']    = int(np.sum(threshold_mask))
        results[f'cluster_rain_total{t_suffix}']   = float(np.nansum(rain[cluster_mask]))
        results[f'cluster_n_cells{t_suffix}']      = int(np.sum(cluster_mask))

        # Flood outcomes summed over each footprint definition
        for depth in [10, 30]:
            for var in ['area', 'vol']:
                fa = flood_arrays[(depth, var)]
                results[f'threshold_flood_{depth}_{var}{t_suffix}'] = float(np.nansum(fa[threshold_mask]))
                results[f'cluster_flood_{depth}_{var}{t_suffix}']   = float(np.nansum(fa[cluster_mask]))


        # ── Optional diagnostic plot, generated once at the 50% level only ──
        if plot and threshold_level == 0.5:
            plot_cluster_check(
                rain=rain, peak_slice=peak_slice,
                threshold_mask=threshold_mask, cluster_mask=cluster_mask,
                row=this_event_results, y0=y0, y1=y1, x0=x0, x1=x1,
                threshold_level=threshold_level,
                neighbourhood_sum=results['neighbourhood_rain_sum_excl_peak'],
                neighbourhood_size=neighbourhood_size,
                flood=flood_cubes[10]['area'][this_event_results['event_num'] - 1],
                catchment_poly=CATCHMENT_POLY, boundary_gdf=boundary_gdf)

    # ── Sanity checks ─────────────────────────────────────────────────────
    # (These use the LAST threshold_level's cluster_mask from the loop above,
    # since Python leaves loop variables bound after the loop exits — a bit
    # implicit; worth being aware this only checks the final threshold level,
    # not all of them.)
    assert cluster_mask[yi, xi] == True             # peak cell must always be part of its own cluster
    assert results['neighbourhood_n_cells'] > 0      # window wasn't entirely out-of-catchment
    assert results['neighbourhood_n_cells'] <= (2*n+1)**2  # window wasn't double-counted/malformed
    assert results['cluster_rain_total_t50'] <= results['threshold_rain_total_t50']  # cluster ⊆ threshold set

    return results

In [3]:
full_rain_cube = get_rainfall_cube_subsection(2015, '01', f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/01/", 1, 2)

from collections import namedtuple
SoilMoisture = namedtuple('SoilMoisture', ['cube', 'data', 'times', 'day_index'])
    
# ── Get list of catchments to run ────────────────────────────────────────────────────────────────────
# Incl. those with both rainfall and flood outputs

all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())
catchments_with_flood_output = []
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = RAIN_ROOT + f"Catchment_{catchment_num}/{catchment_name}.pkl"
    flood_fp = FLOOD_ROOT + f"Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp):
        catchments_with_flood_output.append(catchment_num)    
    

In [4]:
catchment_num = '23'

print(f"Running for {catchment_num}")
if not os.path.exists(os.path.join(OUT_DIR, f"Catchment_{catchment_num}")):
    os.mkdir(os.path.join(OUT_DIR, f"Catchment_{catchment_num}"))

catchment_name  = CATCHMENT_LOOKUP_DICT[str(catchment_num)]        
boundary_gdf    = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
CATCHMENT_POLY = boundary_gdf.geometry.iloc[0] 

# If output already exists, then skip this catchment
combined_fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_new_Oct.pkl")
# if os.path.exists(combined_fp):
#     continue        

# ── Create mask ────────────────────────────────────────────────────────────────────    
mask, mask_sub = build_catchment_mask(full_rain_cube, CATCHMENT_POLY)

results = []



Running for 23


In [5]:
# --------------------------
#  For creating mask 
# --------------------------

catchment_flood_dir    = FLOOD_ROOT + f"Catchment_{catchment_num}/"
flood_numpy = {}
flood_cubes = {}
current_ens = None
DIAGNOSE_EVENT = []

In [6]:
ens_num = '01'

member_fp = os.path.join(OUT_DIR,f"Catchment_{catchment_num}",f"{catchment_name}_EM{ens_num}_new_Oct.pkl")

# if os.path.exists(member_fp):
#     results.append(pd.read_pickle(member_fp)); continue

# --------------------------
# Get rainfall event data, add variables
# Get flood data
# --------------------------
events = add_hydro_year(load_rainfall_events(catchment_name, ens_num))
flood_cubes, flood_numpy = load_flood_arrays(catchment_num, ens_num, mask, CATCHMENT_POLY)   


In [66]:
results_this_ens = []
for hydro_year, events_in_year in events.groupby('hydro_year'):
    # Get soil moisture data for this year
    sm = load_soil_moisture(events_in_year, ens_num, hydro_year, CATCHMENT_POLY, mask)

    # --------------------------
    # Loop through all events within this 'hydro-year'
    # -------------------------
    for row in events_in_year.itertuples():
        print(row.event_num)
        df = process_event(row, ens_num, hydro_year)
        print(row.peaks, df['rainpeak_max_precip'])
        
        # Check rainfall peak consistency
        cube_precip = float(df['rainpeak_max_precip'])
        csv_precip = float(row.peaks)

        df["mismatch"] = not math.isclose(cube_precip, csv_precip, abs_tol=0.001)
        if df["mismatch"]:
            print(f"MISMATCH — ens={ens_num}, year={hydro_year}, event={event_num}")
            print(f"  cube: {cube_precip:.8f}")
            print(f"  csv:  {csv_precip:.8f}")    
            
        results_this_ens.append(df)

1
30.50427 30.504
2
32.84921 32.849
3
56.25648 56.256
4
35.77378 35.774
5
39.58595 39.586
6
59.2145 59.215
7
35.13319 35.133
8
35.98185 35.982
9
50.50747 50.507
10
35.27545 35.275
11
32.86021 32.86
12
31.48035 31.48
13
40.25086 40.251
14
49.58438 49.584
15
37.92081 37.921
16
37.14746 37.147
17
32.84941 32.849
18
36.70547 36.705
19
64.98687 64.987
20
38.66216 38.662
21
41.09405 41.094
22
45.2067 45.207
23
33.8768 33.877
24
35.85451 35.855
25
50.19445 50.194
26
59.27983 59.28
27
32.78268 32.783
28
31.82252 31.823
29
56.48826 56.488
30
40.72715 40.727
31
36.50463 36.505
32
31.63395 31.634
33
34.94739 34.947
34
30.14929 30.149
35
31.17604 31.176
36
48.76494 48.765
37
48.58851 48.589
38
40.70429 40.704
39
55.6582 55.658
40
34.68145 34.681
41
33.6222 33.622
42
32.28646 32.286
43
32.04504 32.045
44
85.87154 85.872
45
59.20121 59.201
46
30.87077 30.871
47
34.83369 34.834
48
32.56445 32.564
49
36.17747 36.177
50
35.42408 35.424
51
45.96746 45.967
52
71.03718 71.037
53
32.45166 32.452
54
30.4165

In [63]:
resutls = pd.read_pickle(OUT_DIR + "Catchment_23/Tyne(Northumberland)_new.pkl")
resutls = resutls[resutls['ens']=='01']
resutls[['ens', 'start_year', 'event_num', 'start_idx', 'stop_idx', "max_precip_from_csv", "start_month", "start_day", "start_hour"]]

,ens,start_year,event_num,start_idx,stop_idx,max_precip_from_csv,start_month,start_day,start_hour
0,01,1992,1,6177,6194,30.50427,8,18,8
1,01,1994,2,5293,5303,32.84921,7,11,12
2,01,1994,3,5384,5411,56.25648,7,15,7
3,01,1995,4,6829,6850,35.77378,9,15,12
4,01,2001,5,4236,4250,39.58595,5,27,11
...,...,...,...,...,...,...,...,...,...
90,01,2078,91,5747,5761,40.01659,7,30,10
91,01,2078,92,5799,5807,43.60483,8,2,14
92,01,2078,93,5956,5976,30.22057,8,9,3
93,01,2078,94,6744,6773,49.28308,9,11,23


In [68]:
old = pd.read_pickle(OUT_DIR + "Catchment_23/Tyne(Northumberland)_new.pkl")
old = old[old['ens'] == '01'].set_index('event_num').sort_index()

new = pd.DataFrame(results_this_ens)          # or pd.read_pickle of the new output
new = new[new['ens'] == '01'].set_index('event_num').sort_index()

print(len(old), len(new), old.index.equals(new.index))   # must be the same events
print("only in old:", sorted(set(old.columns) - set(new.columns)))

rename = {   # old name -> new name (rainfall-peak anchor only; floodpeak_* has no old equivalent)
    'x_idx': 'rainpeak_x_idx', 'y_idx': 'rainpeak_y_idx',
    'x_idx_global': 'rainpeak_x_global', 'y_idx_global': 'rainpeak_y_global',
    'max_precip': 'rainpeak_peak_value', 't_local': 'rainpeak_t_local',
    '10cm_area': 'rainpeak_10cm_area',  '10cm_volume': 'rainpeak_10cm_volume',
    '30cm_area': 'rainpeak_30cm_area',  '30cm_volume': 'rainpeak_30cm_volume',
    'hydro_day_360': 'rainfall_peak_day_hydro_day_360',
    'day_360': 'rainfall_peak_day_daY_360',
}
same = ['start_idx', 'stop_idx', 'start_year', 'start_month', 'start_day', 'start_hour',
        'event_durations', 'max_precip_from_csv', 'time_mismatch', 't_global', 'rainfall_peak_day']
same += [c for c in old.columns if c.startswith(('neighbourhood_', 'threshold_', 'cluster_'))]
col_map = {**{c: c for c in same}, **rename}

# tolerances for columns that are now rounded when stored
atol = {'max_precip': 0.05, '10cm_area': 0.005, '30cm_area': 0.005,
        '10cm_volume': 1.0, '30cm_volume': 1.0}

rows = []
for o, n in col_map.items():
    if o not in old.columns or n not in new.columns:
        rows.append({'old': o, 'new': n, 'status': 'MISSING'}); continue
    a, b = old[o], new[n]
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        a, b = a.astype(float), b.astype(float)
        ok = np.isclose(a, b, rtol=1e-6, atol=atol.get(o, 1e-6), equal_nan=True)
        maxdiff = np.nanmax(np.abs(a - b)) if (~ok).any() else 0.0
    else:
        ok = ((a == b) | (a.isna() & b.isna())).to_numpy()
        maxdiff = np.nan
    rows.append({'old': o, 'new': n, 'status': 'ok' if ok.all() else 'DIFF',
                 'n_diff': int((~ok).sum()), 'max_abs_diff': maxdiff,
                 'first_events': list(a.index[~ok][:5])})

report = pd.DataFrame(rows)
print(report.sort_values('n_diff', ascending=False).to_string())

95 95 True
only in old: ['10cm_area', '10cm_volume', '30cm_area', '30cm_volume', 'cluster_ante_rain_10d_mean_t50', 'cluster_ante_rain_10d_mean_t60', 'cluster_ante_rain_10d_mean_t80', 'cluster_ante_rain_1d_mean_t50', 'cluster_ante_rain_1d_mean_t60', 'cluster_ante_rain_1d_mean_t80', 'cluster_ante_rain_2d_mean_t50', 'cluster_ante_rain_2d_mean_t60', 'cluster_ante_rain_2d_mean_t80', 'cluster_ante_rain_5d_mean_t50', 'cluster_ante_rain_5d_mean_t60', 'cluster_ante_rain_5d_mean_t80', 'cluster_ante_sm_mean_t50', 'cluster_ante_sm_mean_t60', 'cluster_ante_sm_mean_t80', 'd50', 'day_360', 'dry_ratio', 'gini', 'hydro_day_360', 'max_precip', 'max_quartile_index', 'mean_sm_2_before_event', 'mean_sm_3_before_event', 'mean_sm_4_before_event', 'mean_sm_5_before_event', 'neighbourhood_ante_rain_10d_mean', 'neighbourhood_ante_rain_10d_point', 'neighbourhood_ante_rain_1d_mean', 'neighbourhood_ante_rain_1d_point', 'neighbourhood_ante_rain_2d_mean', 'neighbourhood_ante_rain_2d_point', 'neighbourhood_ante_rain_